In [270]:
import numpy as np
import matplotlib.pyplot as plt
import math
import random
import torch


In [336]:
class Value:
    def __init__(self,data ,_children=(),_op =''):
        self.data = data
        self._prev = set(_children)
        self._op = _op
        self._backward = lambda :None
        self.grad =0.0
        self.label =''
    def __repr__(self):
        return f"Value(data:{self.data},grad:{self.grad})"
    
    def __add__(self, other):
        other = other if isinstance(other ,Value) else Value(other)
        out = Value(self.data + other.data,(self,other),'+')
        
        def _backward():
            self.grad += 1.0*out.grad
            other.grad += 1.0*out.grad
        out._backward =_backward
        return out
    
    def __radd__(self, other):
        return self + other
        
    def __sub__(self, other):
        return self + (-other)
    def __rsub__(self, other):
        return self - other
        
    def __neg__(self):
        return -1*self
        
        
    def __mul__(self, other):
        other = other if isinstance(other ,Value) else Value(other)
        out= Value(self.data*other.data,(self,other),'*')
        
        def _backward():
            self.grad += out.grad*other.data
            other.grad += out.grad *self.data
            
            
        out._backward = _backward
        return out
    
    def __rmul__(self, other):
        return self*other
    
    def __truediv__(self, other):
        return self*(other**(-1))
    
    def __pow__(self, other):
        assert isinstance(other ,(int,float))
        out = Value(self.data**other,(self,),f'**{other}')
        
        def _backward():
            self.grad += out.grad*(other*self.data**(other-1))
            
        out._backward=_backward
        return out 
        
    
    def exp(self):
        x= self.data
        out = Value(math.exp(x),(self,),'exp')
        
        def _backward():
            self.grad += out.grad*out.data
        out._backward= _backward
        return out
    
    def tanh(self):
        n= self.data
        t= (math.exp(2*n)-1)/(math.exp(2*n)+1)
        out = Value(t,(self,),'tanh')
        
        def _backward():
            self.grad +=( 1-t**2) *out.grad
            
        out._backward = _backward
        return out 
    
    def backward(self):
        
        topo = []
        visited = set()
        
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
                
        build_topo(self)
        
        
        self.grad =1.0
        
        for node in reversed(topo):
            node._backward()
            
    

            

f =Value(5.0)
k =Value(9.0)
m =Value(3.0)
g= k*m
a =g+f
b= Value(-3)
c= a*b


In [337]:
x=Value(2.0)
y= Value(4.0)
t= x/y
print(t)

Value(data:0.5,grad:0.0)


In [338]:
class Neuron:
    def __init__(self,nin):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1,1))
    
    def __call__(self, x):
        act  = sum((xi*w for (xi,w) in zip(x,self.w)),self.b)
        out = act.tanh()
        return out 
    def parameters(self):
        return self.w +[self.b]

n= Neuron(3)
x =[2.0, 3.0, -1.0]
n(x)
    
        
        

Value(data:-0.998988176906198,grad:0.0)

In [339]:
class Layer:
    def __init__(self,nin,nout):
        self.neuron = [Neuron(nin) for _ in range(nout)]
    
    def __call__(self,x):
        outs = [neuron(x) for neuron in self.neuron]
        return outs[0] if len(outs) ==1 else outs
    
    def parameters(self):
        layer_params = []
        for neuron in self.neuron:
            layer_params.extend(neuron.parameters())
        return layer_params
    
    

In [344]:
x =[2.0, 3.0, -1.0]
l=Layer(3,3)
l(x)

l.parameters()

[Value(data:-0.726810923093349,grad:0.0),
 Value(data:0.6368236624674681,grad:0.0),
 Value(data:-0.8424668706366443,grad:0.0),
 Value(data:-0.15930592631816287,grad:0.0),
 Value(data:-0.6512540830236155,grad:0.0),
 Value(data:-0.7009801226449199,grad:0.0),
 Value(data:0.3539145783126194,grad:0.0),
 Value(data:0.7608534227162944,grad:0.0),
 Value(data:0.9268227685112844,grad:0.0),
 Value(data:0.31044803011043953,grad:0.0),
 Value(data:0.7829102986241163,grad:0.0),
 Value(data:0.2685433497771559,grad:0.0)]

In [341]:
class MLP:
    def __init__(self,nin, nouts):
        sz=[nin] + nouts
        self.layers = [Layer(sz[i],sz[i+1]) for i in range(len(nouts))]
    
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        
        return x 
    
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]
            
        
        
        
        

In [389]:

mlp = MLP(3,[4,4,1])
xs = [
[2.0, 3.0, -1.0],
[3.0, -1.0, 0.5],
[0.5, 1.0, 1.0],
[1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]


In [390]:
def mse(ypred , ytarget):
    mse = sum((yp-yt)**2 for (yp,yt) in zip(ypred,ytarget))
    return mse

In [395]:
for k in range(1000):
    ypred =[mlp(x) for x in xs]
    loss = mse(ypred,ys)
    for p in mlp.parameters():
        p.grad =0.0
    loss.backward()
    params = mlp.parameters()
    print(f"step: {k} , loss {loss}")
    for p in params:
        
        p.data += (-0.01*p.grad)
ypred =[mlp(x) for x in xs]
print(ypred)

step: 0 , loss Value(data:0.00625021676822757,grad:1.0)
step: 1 , loss Value(data:0.006228596594186601,grad:1.0)
step: 2 , loss Value(data:0.006207122569538539,grad:1.0)
step: 3 , loss Value(data:0.006185793231297452,grad:1.0)
step: 4 , loss Value(data:0.00616460713582425,grad:1.0)
step: 5 , loss Value(data:0.006143562858509211,grad:1.0)
step: 6 , loss Value(data:0.006122658993460815,grad:1.0)
step: 7 , loss Value(data:0.006101894153200525,grad:1.0)
step: 8 , loss Value(data:0.006081266968363735,grad:1.0)
step: 9 , loss Value(data:0.006060776087406129,grad:1.0)
step: 10 , loss Value(data:0.006040420176316038,grad:1.0)
step: 11 , loss Value(data:0.006020197918332275,grad:1.0)
step: 12 , loss Value(data:0.006000108013667287,grad:1.0)
step: 13 , loss Value(data:0.005980149179235611,grad:1.0)
step: 14 , loss Value(data:0.005960320148387675,grad:1.0)
step: 15 , loss Value(data:0.005940619670648349,grad:1.0)
step: 16 , loss Value(data:0.005921046511460894,grad:1.0)
step: 17 , loss Value(data

In [ ]:
# from graphviz import Digraph

# def trace(root):
#     # builds a set of all nodes and edges in a graph
#     nodes, edges = set(), set()
#     def build(v):
#         if v not in nodes:
#             nodes.add(v)
#             for child in v._prev:
#                 edges.add((child, v))
#                 build(child)
#     build(root)
#     return nodes, edges

# def draw_dot(root):
#     dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right

#     nodes, edges = trace(root)
#     for n in nodes:
#         uid = str(id(n))
#         # for any value in the graph, create a rectangular ('record') node for it
#         dot.node(name = uid, label = "{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')
#         if n._op:
#         # if this value is a result of some operation, create an op node for it
#             dot.node(name = uid + n._op, label = n._op)
#             # and connect this node to it
#             dot.edge(uid + n._op, uid)

#     for n1, n2 in edges:
#         # connect n1 to the op node of n2
#         dot.edge(str(id(n1)), str(id(n2)) + n2._op)

#     return dot